In [2]:
from ipywidgets import VBox, HBox, Button, FileUpload, Output, Label, Layout, HTML
from PIL import Image
from IPython.display import display
import io
import numpy as np
import torchvision.transforms as T
import torch
import os

# Load model
model_path = "checkpoints/transfer_exported.pt"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found at {model_path}")
learn_inf = torch.jit.load(model_path)
from src.data import get_data_loaders
learn_inf.class_names = get_data_loaders(batch_size=1)["train"].dataset.classes

# Widgets
out_pl = Output()
labels = [Label() for _ in range(5)]
btn_upload = FileUpload(accept='image/*', multiple=False)
btn_run = Button(description="Classify", button_style='success', layout=Layout(width='150px'))

# Classification logic
def on_click_classify(change):
    out_pl.clear_output()
    if not btn_upload.value:
        with out_pl: print("Please upload an image.")
        return
    try:
        upload_dict = (btn_upload.value[0] if isinstance(btn_upload.value, tuple)
                       else list(btn_upload.value.values())[0])
        img_bytes = upload_dict['content']
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        with out_pl:
            thumb = img.copy()
            ratio = thumb.width / thumb.height
            thumb.thumbnail((ratio * 200, 200))
            display(thumb)

        timg = T.ToTensor()(img).unsqueeze(0)
        preds = learn_inf(timg).detach().cpu().numpy().squeeze()
        top_idxs = np.argsort(preds)[::-1]

        for i in range(5):
            idx = top_idxs[i]
            labels[i].value = f"{i+1}. {learn_inf.class_names[idx]} (confidence: {preds[idx]:.2f})"

    except Exception as e:
        with out_pl: print("❌ Error:", e)

btn_run.on_click(on_click_classify)

# UI Layout
title = HTML("<h2 style='color:#2c3e50; text-align:center; margin-top:10px;'>🏛️ Landmark Image Classifier</h2>")
upload_controls = HBox([btn_upload, btn_run], layout=Layout(justify_content="center", gap="10px", margin="10px 0"))
results = VBox(labels, layout=Layout(border="1px solid #ccc", padding="10px", margin="10px 0", width="100%"))

app_box = VBox([
    title,
    upload_controls,
    out_pl,
    results
], layout=Layout(
    align_items="center",
    border="2px solid #2c3e50",
    padding="20px",
    width="500px",
    background_color="#f9f9f9"
))

VBox([app_box], layout=Layout(align_items="center", justify_content="center"))


Reusing cached mean and std
Dataset mean: tensor([0.4638, 0.4725, 0.4687]), std: tensor([0.2697, 0.2706, 0.3017])
